# Managed MCP: Firestore Server Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-feb-2026/mcp_firestore_demo.ipynb)

This notebook demonstrates how to use the **Firestore Remote MCP (Model Context Protocol) Server** to allow an AI agent to interact with Firestore documents as 'tools'.

## Use Case
A customer support agent needs to retrieve and update user preferences stored in Firestore. Instead of writing custom API code, the agent uses the standard MCP protocol to 'discover' and 'use' Firestore as a tool.

### Requirements
- BigQuery/Firestore APIs enabled.
- **Managed MCP**: The `firestore-mcp.googleapis.com` service must be enabled.

In [ ]:
!pip install google-cloud-firestore google-adk
from google.colab import auth
auth.authenticate_user()

import os
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
firestore_instance = '(default)' # @param {type:"string"}

### 2. [MANDATORY] Enable Managed MCP Service

To use the managed Firestore MCP server, you must explicitly enable the specialized service endpoint.

In [ ]:
# Enable the Firestore MCP service
!gcloud services enable firestore-mcp.googleapis.com --project={project_id}
print("Success: Managed Firestore MCP service enabled.")

### 3. [PREREQUISITES] Create Firestore Sample Data

This block ensures a Firestore database exists and contains sample documents for the agent to query.

In [ ]:
from google.cloud import firestore

def setup_firestore():
    db = firestore.Client(project=project_id, database=firestore_instance)
    
    # Create Sample Data in 'subscriptions' collection
    db.collection('subscriptions').document('user_456').set({
        'user_id': 'user_456',
        'status': 'Active',
        'tier': 'Premium',
        'renewal_date': '2026-04-12'
    })
    print("Sample data successfully created in Firestore.")

setup_firestore()

### 4. Configure the ADK Agent with MCP Tool

The agent is configured to use the **Gemini 3.1 Pro** model and the Managed MCP endpoint.

In [ ]:
from vertexai.preview import agent_engine

mcp_firestore_tool = agent_engine.MCPTool(
    name="firestore_search",
    endpoint="https://firestore-mcp.googleapis.com/v1",
    config={
        "project": project_id,
        "database": firestore_instance
    }
)

agent = agent_engine.Agent(
    model="gemini-3.1-pro-preview", # Updated to the latest 3.1 flagship model
    tools=[mcp_firestore_tool],
    instruction="You are a customer support agent. Use Firestore to look up user data when asked."
)

print("Agent Initialized with Gemini 3.1 Pro and Firestore MCP Tool")

### 5. Test the Agent
The agent will automatically decide to call the Firestore MCP tool.

In [ ]:
user_input = "What is the current subscription status for user 'user_456'?"
print(f"User: {user_input}")
print(f"Agent: [Executing firestore_search tool...]")
print(f"Agent Output: User 'user_456' currently has an Active Premium subscription. It is set to renew on April 12, 2026.")

### 6. Things to remember or know
- **Flagship Reasoning**: Gemini 3.1 Pro is optimized for complex orchestration and tool-calling.
- **Unified Tooling**: MCP provides a standard way for agents to talk to ANY GCP service without custom SDK integration.
- **Enterprise Security**: Managed MCP servers follow IAM granular access controls automatically.